# Valuation-to-Option Pricing Lab

This notebook is the analytical engine for the student portfolio dashboard. It performs the valuation analysis, validates the option-pricing model, exports `docs/model_results.json`, and supplies the audit trail behind the published dashboard.

> **Theoretical boundary:** Analyst targets inform beliefs about terminal equity values. They are not risk-neutral probabilities. The CRR lattice independently produces the no-arbitrage option value.

## 1. Imports and controlled model inputs

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data"
DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

MODEL = {
    "company": "Asteron Systems",
    "ticker": "ASTR",
    "valuation_date": "2026-10-16",
    "spot": 96.00,
    "strike": 105.00,
    "maturity_years": 1.00,
    "risk_free_rate": 0.045,
    "dividend_yield": 0.012,
    "volatility": 0.280,
    "option_type": "call",
    "exercise_style": "european",
    "lattice_steps": 200,
    "market_bid": 8.10,
    "market_ask": 8.60,
    "dcf_value": 106.00,
    "comps_value": 101.00,
}
MODEL["market_mid"] = (MODEL["market_bid"] + MODEL["market_ask"]) / 2
MODEL

## 2. Equity valuation and analyst target distribution

In [ ]:
targets = pd.read_csv(DATA / "analyst_targets.csv")
required = {"analyst", "target", "publication_date", "horizon_months"}
assert required.issubset(targets.columns)
assert targets["target"].gt(0).all()
assert targets["horizon_months"].nunique() == 1, "Analyst horizons must be aligned."

analyst_stats = {
    "count": int(targets["target"].count()),
    "mean": float(targets["target"].mean()),
    "median": float(targets["target"].median()),
    "std": float(targets["target"].std(ddof=1)),
    "min": float(targets["target"].min()),
    "max": float(targets["target"].max()),
}
valuation = {
    "Current market": MODEL["spot"],
    "Comparable companies": MODEL["comps_value"],
    "Student DCF": MODEL["dcf_value"],
    "Analyst median": analyst_stats["median"],
}
pd.Series(valuation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(valuation.keys(), valuation.values(), color=["#66758a", "#d9a23a", "#216b8f", "#18a59b"])
axes[0].set_title("Equity valuation comparison")
axes[0].set_ylabel("Value per share ($)")
axes[0].tick_params(axis="x", rotation=20)
axes[1].scatter(targets["target"], np.zeros(len(targets)), s=70, color="#18a59b")
axes[1].axvline(analyst_stats["mean"], color="#10243f", label="Mean")
axes[1].axvline(analyst_stats["median"], color="#d9a23a", linestyle="--", label="Median")
axes[1].set_yticks([])
axes[1].set_title("Analyst target distribution")
axes[1].set_xlabel("12-month target ($)")
axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Subjective valuation states

In [ ]:
states = pd.DataFrame({
    "state": ["Bear", "Base", "Bull", "Tail"],
    "terminal_price": [82.0, 101.0, 116.0, 135.0],
    "subjective_weight": [0.20, 0.45, 0.25, 0.10],
})
assert np.isclose(states["subjective_weight"].sum(), 1.0)
assert states["subjective_weight"].ge(0).all()
state_mean = float(np.average(states["terminal_price"], weights=states["subjective_weight"]))
mean_preservation_gap = state_mean - analyst_stats["mean"]
states

## 4. State-contingent derivative payoff

In [ ]:
def payoff(stock, strike, option_type="call"):
    stock = np.asarray(stock, dtype=float)
    if option_type == "call":
        return np.maximum(stock - strike, 0.0)
    if option_type == "put":
        return np.maximum(strike - stock, 0.0)
    raise ValueError("option_type must be call or put")

states["option_payoff"] = payoff(states["terminal_price"], MODEL["strike"], MODEL["option_type"])
states["weighted_payoff"] = states["subjective_weight"] * states["option_payoff"]
valuation_weighted = float(np.exp(-MODEL["risk_free_rate"] * MODEL["maturity_years"]) * states["weighted_payoff"].sum())
states

## 5. CRR risk-neutral lattice

In [ ]:
def crr_option_price(spot, strike, maturity, rate, volatility, steps=200,
                     dividend_yield=0.0, option_type="call", american=False):
    dt = maturity / steps
    u = np.exp(volatility * np.sqrt(dt))
    d = 1 / u
    growth = np.exp((rate - dividend_yield) * dt)
    p_star = (growth - d) / (u - d)
    discount = np.exp(-rate * dt)
    if not (d < growth < u and 0 < p_star < 1):
        raise ValueError("No-arbitrage condition failed")
    j = np.arange(steps + 1)
    stock = spot * (u ** j) * (d ** (steps - j))
    values = payoff(stock, strike, option_type)
    for n in range(steps - 1, -1, -1):
        values = discount * (p_star * values[1:n+2] + (1-p_star) * values[:n+1])
        if american:
            j = np.arange(n + 1)
            stock = spot * (u ** j) * (d ** (n-j))
            values = np.maximum(values, payoff(stock, strike, option_type))
    return {"price": float(values[0]), "u": float(u), "d": float(d), "p_star": float(p_star), "dt": float(dt)}

lattice = crr_option_price(
    MODEL["spot"], MODEL["strike"], MODEL["maturity_years"], MODEL["risk_free_rate"],
    MODEL["volatility"], MODEL["lattice_steps"], MODEL["dividend_yield"],
    MODEL["option_type"], MODEL["exercise_style"] == "american"
)
lattice

## 6. Automated theoretical validation

In [ ]:
def european_bounds(spot, strike, maturity, rate, dividend_yield, option_type):
    pv_s = spot * np.exp(-dividend_yield*maturity)
    pv_k = strike * np.exp(-rate*maturity)
    return (max(pv_s-pv_k,0), pv_s) if option_type == "call" else (max(pv_k-pv_s,0), pv_k)

lower, upper = european_bounds(MODEL["spot"], MODEL["strike"], MODEL["maturity_years"], MODEL["risk_free_rate"], MODEL["dividend_yield"], MODEL["option_type"])
call = crr_option_price(MODEL["spot"],MODEL["strike"],MODEL["maturity_years"],MODEL["risk_free_rate"],MODEL["volatility"],MODEL["lattice_steps"],MODEL["dividend_yield"],"call")["price"]
put = crr_option_price(MODEL["spot"],MODEL["strike"],MODEL["maturity_years"],MODEL["risk_free_rate"],MODEL["volatility"],MODEL["lattice_steps"],MODEL["dividend_yield"],"put")["price"]
parity_rhs = MODEL["spot"]*np.exp(-MODEL["dividend_yield"]*MODEL["maturity_years"]) - MODEL["strike"]*np.exp(-MODEL["risk_free_rate"]*MODEL["maturity_years"])
parity_residual = float((call-put)-parity_rhs)
steps_grid = [25,50,100,200,500]
convergence = [crr_option_price(MODEL["spot"],MODEL["strike"],MODEL["maturity_years"],MODEL["risk_free_rate"],MODEL["volatility"],n,MODEL["dividend_yield"],MODEL["option_type"])["price"] for n in steps_grid]
validation = {
    "valid_risk_neutral_probability": bool(0 < lattice["p_star"] < 1),
    "inside_theoretical_bounds": bool(lower <= lattice["price"] <= upper),
    "put_call_parity_pass": bool(abs(parity_residual) < 0.05),
    "convergence_pass": bool(abs(convergence[-1]-convergence[-2]) < 0.05),
    "parity_residual": parity_residual,
    "convergence_gap_200_500": float(convergence[-1]-convergence[-2]),
}
assert all(validation[k] for k in ["valid_risk_neutral_probability","inside_theoretical_bounds","put_call_parity_pass","convergence_pass"])
validation

## 7. Export dashboard data

In [ ]:
results = {
    "model": MODEL,
    "analyst_targets": [float(x) for x in targets["target"]],
    "analyst_stats": analyst_stats,
    "valuation": {k: float(v) for k,v in valuation.items()},
    "states": states.to_dict(orient="records"),
    "state_mean": state_mean,
    "mean_preservation_gap": float(mean_preservation_gap),
    "valuation_weighted_expected_payoff": valuation_weighted,
    "lattice": lattice,
    "validation": validation,
    "convergence": [{"steps": n, "price": float(v)} for n,v in zip(steps_grid,convergence)],
    "comparison": {
        "Valuation-weighted expected payoff": valuation_weighted,
        "Risk-neutral lattice value": lattice["price"],
        "Observed market midpoint": MODEL["market_mid"],
    },
}
(DOCS / "model_results.json").write_text(json.dumps(results, indent=2))
print(f"Exported {DOCS / 'model_results.json'}")
results["comparison"]

## 8. Decision memo
The analyst-weighted discounted payoff is not presented as an arbitrage-free option price. The CRR lattice uses market-consistent inputs and passes no-arbitrage, bounds, put-call parity, and convergence checks. The dashboard reads the exported JSON produced by this notebook.